# Parakeet-TDT 0.6B: Audio Feature Extraction + XGBoost

1. Extract audio embeddings from NVIDIA Parakeet-TDT 0.6B (CPU inference)
2. Train XGBoost on parakeet embeddings (80% train)
3. Train XGBoost on text features (same 80% split)
4. Combine parakeet + text XGBoost + wav2vec2 with optimal weights
5. Threshold sweep for best F1

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import xgboost as xgb
import joblib

# ── Paths ──────────────────────────────────────────────────
FEATURES_CSV = "features_company.csv"       # existing text+pause+prosodic features
PARAKEET_CSV = "features_parakeet.csv"       # will be created by this notebook
ONNX_MODEL = None  # set below via auto-detect

# Auto-detect wav2vec2 ONNX
for candidate in [
    "wav2vec2_combined_quant.onnx", "models/wav2vec2_combined_quant.onnx",
    "biased_wav2vec2_quant.onnx", "models/biased_wav2vec2_quant.onnx",
    "checkpoints_combined/wav2vec2_combined_quant.onnx",
    "checkpoints_biased/biased_wav2vec2_quant.onnx",
]:
    if os.path.exists(candidate):
        ONNX_MODEL = candidate
        break

print(f"Features CSV:  {FEATURES_CSV}")
print(f"Parakeet CSV:  {PARAKEET_CSV}")
print(f"wav2vec2 ONNX: {ONNX_MODEL or 'NOT FOUND'}")

## 1. Extract Parakeet Embeddings (CPU)

Uses `nemo.collections.asr` to load parakeet-tdt-0.6b and extract encoder embeddings.
Each audio file → mean-pooled 512-dim embedding vector.

In [ ]:
# Install NeMo toolkit (CPU-only if no GPU)
!pip install -q nemo_toolkit[asr] librosa soundfile

In [ ]:
import torch
import librosa
import nemo.collections.asr as nemo_asr
from tqdm import tqdm

# Load features CSV to get file paths
df = pd.read_csv(FEATURES_CSV)
df = df[df["label_int"].isin([0, 1])].reset_index(drop=True)
print(f"Loaded {len(df)} labelled samples")

# Load parakeet model on CPU
print("Loading parakeet-tdt-0.6b (this may take a minute)...")
asr_model = nemo_asr.models.ASRModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v2")
asr_model = asr_model.eval()
asr_model = asr_model.to("cpu")
print("Model loaded on CPU.")

# Get encoder embedding dim
# Parakeet-TDT uses FastConformer encoder → output dim is 512
EMBED_DIM = asr_model.encoder.d_model if hasattr(asr_model.encoder, 'd_model') else 512
print(f"Encoder embedding dim: {EMBED_DIM}")

In [ ]:
SR = 16000
MAX_DURATION = 30  # cap at 30s for CPU memory/speed, take first 30s of longer files

embeddings = []
failed = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting parakeet embeddings"):
    fp = row.get("filepath", "")
    if not fp or not os.path.exists(fp):
        embeddings.append(np.zeros(EMBED_DIM))
        failed.append(idx)
        continue
    try:
        audio, _ = librosa.load(fp, sr=SR, mono=True, duration=MAX_DURATION)
        if len(audio) < SR:  # less than 1 second
            embeddings.append(np.zeros(EMBED_DIM))
            failed.append(idx)
            continue

        # Run encoder only (no decoder/transcription needed)
        audio_tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)  # (1, T)
        audio_len = torch.tensor([len(audio)], dtype=torch.int64)

        with torch.no_grad():
            encoded, encoded_len = asr_model.encoder(
                audio_signal=audio_tensor, length=audio_len
            )
            # encoded shape: (1, T_enc, D)
            # Mean-pool over time dimension
            seq_len = encoded_len[0].item()
            emb = encoded[0, :seq_len, :].mean(dim=0).numpy()  # (D,)

        embeddings.append(emb)
    except Exception as e:
        print(f"  Failed {fp}: {e}")
        embeddings.append(np.zeros(EMBED_DIM))
        failed.append(idx)

print(f"\nExtracted {len(embeddings)} embeddings, {len(failed)} failed")

# Build dataframe
embed_cols = [f"parakeet_{i}" for i in range(EMBED_DIM)]
embed_df = pd.DataFrame(embeddings, columns=embed_cols)
embed_df["filename"] = df["filename"]
embed_df["filepath"] = df["filepath"]
embed_df["label_int"] = df["label_int"]
if "audio_batch" in df.columns:
    embed_df["audio_batch"] = df["audio_batch"]

embed_df.to_csv(PARAKEET_CSV, index=False)
print(f"Saved: {PARAKEET_CSV} ({len(embed_df)} rows x {len(embed_cols)} features)")

## 2. Load All Features

Load text features (original 41) + parakeet embeddings + wav2vec2 scores.
Create a shared 80/20 train/test split.

In [ ]:
# Load or reload
df_text = pd.read_csv(FEATURES_CSV)
df_text = df_text[df_text["label_int"].isin([0, 1])].reset_index(drop=True)

df_para = pd.read_csv(PARAKEET_CSV)

# Verify alignment
assert len(df_text) == len(df_para), f"Row mismatch: text={len(df_text)}, parakeet={len(df_para)}"
print(f"Loaded {len(df_text)} samples")

y = df_text["label_int"].values

# ── Original 41 text features ──────────────────────────────
TEXT_FEATURES = [
    "filler_rate", "filler_count", "repetition_rate", "repair_rate",
    "ttr", "mattr", "complex_word_rate", "avg_word_length",
    "n_words", "n_unique_words",
    "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
    "self_ref_rate", "discourse_marker_rate", "hedge_rate",
    "noun_rate", "verb_rate", "adj_rate",
]
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
TEXT_ONLY_FEATURES = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES  # 41

# Filter to available
text_cols = [c for c in TEXT_ONLY_FEATURES if c in df_text.columns]
print(f"Text features: {len(text_cols)}/41 available")

# ── Parakeet embedding columns ────────────────────────────
para_cols = [c for c in df_para.columns if c.startswith("parakeet_")]
print(f"Parakeet features: {len(para_cols)}")

# ── Shared train/test split (80/20, stratified) ───────────
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(idx_train)} (cheating={y[idx_train].sum()})")
print(f"Test:  {len(idx_test)} (cheating={y[idx_test].sum()})")

if "audio_batch" in df_text.columns:
    for name, idxs in [("Train", idx_train), ("Test", idx_test)]:
        counts = df_text.iloc[idxs]["audio_batch"].value_counts().to_dict()
        print(f"  {name} batches: {counts}")

## 3. Train Parakeet XGBoost (audio embeddings only)

In [ ]:
X_para = df_para[para_cols].fillna(0).values

X_para_train, X_para_test = X_para[idx_train], X_para[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

para_scaler = StandardScaler()
X_para_train_s = para_scaler.fit_transform(X_para_train)
X_para_test_s = para_scaler.transform(X_para_test)

para_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.3,  # lower since 512 features — reduce overfitting
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

para_model.fit(
    X_para_train_s, y_train,
    eval_set=[(X_para_train_s, y_train), (X_para_test_s, y_test)],
    verbose=20,
)

para_proba_test = para_model.predict_proba(X_para_test_s)[:, 1]
para_preds_test = (para_proba_test >= 0.5).astype(int)

print(f"\n--- Parakeet XGBoost (test set) ---")
print(f"Accuracy: {accuracy_score(y_test, para_preds_test):.4f}")
print(f"F1:       {f1_score(y_test, para_preds_test, zero_division=0):.4f}")
print(classification_report(y_test, para_preds_test, target_names=["not cheating", "cheating"]))

## 4. Train Text XGBoost (original 41 features, same split)

In [ ]:
X_text_all = df_text[text_cols].fillna(0).values

X_text_train, X_text_test = X_text_all[idx_train], X_text_all[idx_test]

text_scaler = StandardScaler()
X_text_train_s = text_scaler.fit_transform(X_text_train)
X_text_test_s = text_scaler.transform(X_text_test)

text_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

text_model.fit(
    X_text_train_s, y_train,
    eval_set=[(X_text_train_s, y_train), (X_text_test_s, y_test)],
    verbose=20,
)

text_proba_test = text_model.predict_proba(X_text_test_s)[:, 1]
text_preds_test = (text_proba_test >= 0.5).astype(int)

print(f"\n--- Text XGBoost (test set) ---")
print(f"Accuracy: {accuracy_score(y_test, text_preds_test):.4f}")
print(f"F1:       {f1_score(y_test, text_preds_test, zero_division=0):.4f}")
print(classification_report(y_test, text_preds_test, target_names=["not cheating", "cheating"]))

## 5. wav2vec2 Scores

Load existing wav2vec2 scores from features CSV, or recompute from ONNX.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

has_w2v = "wav2vec2_mean_p_read" in df_text.columns and df_text["wav2vec2_mean_p_read"].sum() > 0

if has_w2v:
    w2v_proba = df_text["wav2vec2_mean_p_read"].values
    print(f"Using existing wav2vec2 scores from {FEATURES_CSV}")
elif ONNX_MODEL:
    import onnxruntime as ort
    print(f"Computing wav2vec2 scores from {ONNX_MODEL}...")
    sess = ort.InferenceSession(ONNX_MODEL, providers=["CPUExecutionProvider"])
    SR = 16000
    WINDOW = 5 * SR
    w2v_proba = np.zeros(len(df_text))
    for i, row in tqdm(df_text.iterrows(), total=len(df_text), desc="wav2vec2"):
        fp = row.get("filepath", "")
        if not fp or not os.path.exists(fp):
            continue
        try:
            audio, _ = librosa.load(fp, sr=SR, mono=True)
        except Exception:
            continue
        if len(audio) < WINDOW:
            continue
        p_reads = []
        for start in range(0, len(audio) - WINDOW + 1, WINDOW):
            chunk = audio[start:start+WINDOW].astype(np.float32).reshape(1, -1)
            logits = sess.run(None, {"input_values": chunk})[0]
            p_reads.append(sigmoid(logits.flatten()[0]))
        if p_reads:
            w2v_proba[i] = float(np.mean(p_reads))
else:
    print("WARNING: No wav2vec2 scores available. Setting to 0.")
    w2v_proba = np.zeros(len(df_text))

w2v_test = w2v_proba[idx_test]
w2v_preds_test = (w2v_test >= 0.5).astype(int)
print(f"\nwav2vec2 test F1: {f1_score(y_test, w2v_preds_test, zero_division=0):.4f}")
print(f"  mean={w2v_proba.mean():.4f}, std={w2v_proba.std():.4f}")

## 6. Combined Model: Optimal Weights for 3 Voters

Three independent models:
- **Parakeet XGBoost** (audio embeddings)
- **Text XGBoost** (41 text+pause+prosodic features)
- **wav2vec2** (read probability)

Grid search over weight triplets (w_para, w_text, w_w2v) that sum to 1.

In [ ]:
print("="*60)
print("3-MODEL WEIGHT SEARCH (test set)")
print("="*60)

# Individual model scores on test set
print(f"\nIndividual models:")
print(f"  Parakeet XGBoost: F1={f1_score(y_test, para_preds_test, zero_division=0):.4f}")
print(f"  Text XGBoost:     F1={f1_score(y_test, text_preds_test, zero_division=0):.4f}")
print(f"  wav2vec2:         F1={f1_score(y_test, w2v_preds_test, zero_division=0):.4f}")

# Grid search over weight triplets (step 0.1)
best_f1, best_weights = 0, (0, 0, 0)
results = []

for w_para in np.arange(0, 1.01, 0.1):
    for w_text in np.arange(0, 1.01 - w_para, 0.1):
        w_w2v = round(1.0 - w_para - w_text, 1)
        if w_w2v < 0:
            continue
        combined = w_para * para_proba_test + w_text * text_proba_test + w_w2v * w2v_test
        preds = (combined >= 0.5).astype(int)
        f = f1_score(y_test, preds, zero_division=0)
        results.append({
            "w_para": round(w_para, 1),
            "w_text": round(w_text, 1),
            "w_w2v": round(w_w2v, 1),
            "f1": round(f, 4),
            "acc": round(accuracy_score(y_test, preds), 4),
        })
        if f > best_f1:
            best_f1 = f
            best_weights = (round(w_para, 1), round(w_text, 1), round(w_w2v, 1))

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(f"\nTop 10 weight combinations:")
print(results_df.head(10).to_string(index=False))

W_PARA, W_TEXT, W_W2V = best_weights
print(f"\nBest: para={W_PARA}, text={W_TEXT}, w2v={W_W2V} -> F1={best_f1:.4f}")

## 7. Threshold Sweep

In [ ]:
best_combined = W_PARA * para_proba_test + W_TEXT * text_proba_test + W_W2V * w2v_test

thresholds = np.arange(0.10, 0.91, 0.05)
rows_t = []
for t in thresholds:
    preds = (best_combined >= t).astype(int)
    rows_t.append({
        "threshold": round(t, 2),
        "precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "f1": round(f1_score(y_test, preds, zero_division=0), 4),
        "flagged": int(preds.sum()),
        "missed": int(((y_test == 1) & (preds == 0)).sum()),
        "false_alarms": int(((y_test == 0) & (preds == 1)).sum()),
    })

thresh_df = pd.DataFrame(rows_t)
best_row = thresh_df.loc[thresh_df["f1"].idxmax()]

print(f"Threshold sweep (para={W_PARA}, text={W_TEXT}, w2v={W_W2V})")
print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'flagged':>8s} {'missed':>7s} {'false_alarm':>11s}")
print("-" * 62)
for _, r in thresh_df.iterrows():
    marker = " <-- best F1" if r["threshold"] == best_row["threshold"] else ""
    print(f"  {r['threshold']:.2f}   {r['precision']:.4f}  {r['recall']:.4f}  {r['f1']:.4f}  {r['flagged']:>6d}  {r['missed']:>6d}  {r['false_alarms']:>6d}{marker}")

CHOSEN_THRESHOLD = best_row["threshold"]
print(f"\nBest F1 at threshold={CHOSEN_THRESHOLD:.2f}: F1={best_row['f1']:.4f}")

## 8. 5-Fold Cross-Validation

In [ ]:
X_para_all = df_para[para_cols].fillna(0).values
X_text_feat = df_text[text_cols].fillna(0).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_para_all, y), 1):
    # Parakeet XGBoost
    sc_p = StandardScaler()
    X_p_tr = sc_p.fit_transform(X_para_all[tr_idx])
    X_p_te = sc_p.transform(X_para_all[te_idx])
    m_p = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.3,
        scale_pos_weight=(y[tr_idx]==0).sum() / max((y[tr_idx]==1).sum(), 1),
        eval_metric="logloss", random_state=42,
    )
    m_p.fit(X_p_tr, y[tr_idx], verbose=False)
    para_prob = m_p.predict_proba(X_p_te)[:, 1]

    # Text XGBoost
    sc_t = StandardScaler()
    X_t_tr = sc_t.fit_transform(X_text_feat[tr_idx])
    X_t_te = sc_t.transform(X_text_feat[te_idx])
    m_t = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y[tr_idx]==0).sum() / max((y[tr_idx]==1).sum(), 1),
        eval_metric="logloss", random_state=42,
    )
    m_t.fit(X_t_tr, y[tr_idx], verbose=False)
    text_prob = m_t.predict_proba(X_t_te)[:, 1]

    # wav2vec2
    w2v_fold = w2v_proba[te_idx]

    # Combined
    combined = W_PARA * para_prob + W_TEXT * text_prob + W_W2V * w2v_fold
    preds = (combined >= CHOSEN_THRESHOLD).astype(int)

    f = f1_score(y[te_idx], preds, zero_division=0)
    p = precision_score(y[te_idx], preds, zero_division=0)
    r = recall_score(y[te_idx], preds, zero_division=0)
    fold_results.append({"fold": fold, "f1": f, "precision": p, "recall": r})
    print(f"  Fold {fold}: F1={f:.4f}  Prec={p:.4f}  Rec={r:.4f}")

fold_df = pd.DataFrame(fold_results)
print(f"\n  Mean:  F1={fold_df['f1'].mean():.4f} +/- {fold_df['f1'].std():.4f}")
print(f"         Prec={fold_df['precision'].mean():.4f} +/- {fold_df['precision'].std():.4f}")
print(f"         Rec={fold_df['recall'].mean():.4f} +/- {fold_df['recall'].std():.4f}")

## 9. Save Models

In [ ]:
SAVE_DIR = "checkpoints_parakeet"
os.makedirs(SAVE_DIR, exist_ok=True)

para_model.save_model(f"{SAVE_DIR}/xgboost_parakeet.json")
joblib.dump(para_scaler, f"{SAVE_DIR}/scaler_parakeet.pkl")

text_model.save_model(f"{SAVE_DIR}/xgboost_text.json")
joblib.dump(text_scaler, f"{SAVE_DIR}/scaler_text.pkl")

config = {
    "text_feature_columns": text_cols,
    "parakeet_feature_columns": para_cols,
    "weights": {"parakeet": W_PARA, "text": W_TEXT, "wav2vec2": W_W2V},
    "threshold": float(CHOSEN_THRESHOLD),
    "test_f1": round(best_f1, 4),
    "cv_f1_mean": round(fold_df["f1"].mean(), 4),
    "n_train": len(idx_train),
    "n_test": len(idx_test),
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved to {SAVE_DIR}/:")
print(f"  xgboost_parakeet.json + scaler_parakeet.pkl")
print(f"  xgboost_text.json     + scaler_text.pkl")
print(f"  results.json")

## 10. Predict on All Data

In [ ]:
# Score all samples with all 3 models
para_all_proba = para_model.predict_proba(para_scaler.transform(X_para_all))[:, 1]
text_all_proba = text_model.predict_proba(text_scaler.transform(X_text_feat))[:, 1]

combined_all = W_PARA * para_all_proba + W_TEXT * text_all_proba + W_W2V * w2v_proba

df_text["parakeet_score"] = para_all_proba
df_text["text_xgb_score"] = text_all_proba
df_text["w2v_score"] = w2v_proba
df_text["combined_score"] = combined_all
df_text["pred_label"] = (combined_all >= CHOSEN_THRESHOLD).astype(int)
df_text["pred_label_str"] = df_text["pred_label"].map({1: "cheating", 0: "not cheating"})

print(f"Predictions ({len(df_text)} files):")
print(f"  Cheating:     {(df_text['pred_label']==1).sum()}")
print(f"  Not cheating: {(df_text['pred_label']==0).sum()}")
print(f"  Weights: para={W_PARA}, text={W_TEXT}, w2v={W_W2V}")
print(f"  Threshold: {CHOSEN_THRESHOLD:.2f}")

# Error analysis
if "label_int" in df_text.columns:
    wrong = df_text[df_text["label_int"] != df_text["pred_label"]]
    fp = wrong[wrong["pred_label"] == 1]
    fn = wrong[wrong["pred_label"] == 0]
    print(f"\nMisclassifications: {len(wrong)} ({len(fp)} FP, {len(fn)} FN)")
    if len(wrong) > 0:
        show_cols = ["filename", "label_int", "pred_label_str", "combined_score",
                     "parakeet_score", "text_xgb_score", "w2v_score"]
        show_cols = [c for c in show_cols if c in wrong.columns]
        print(wrong[show_cols].to_string(index=False))

# Save predictions
out_cols = ["filename", "filepath", "label_int", "pred_label_str", "combined_score",
            "parakeet_score", "text_xgb_score", "w2v_score"]
if "audio_batch" in df_text.columns:
    out_cols.insert(2, "audio_batch")
out_cols = [c for c in out_cols if c in df_text.columns]
df_text[out_cols].to_csv("predictions_parakeet.csv", index=False)
print(f"\nSaved: predictions_parakeet.csv")